## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
os.getcwd()
output_file = '/Users/E2074/local-datasets/customer/ads-monetisation/jas-plan'

## Datasets & Parameter

In [5]:
## Parameter 
start_date = '20240420'
end_date = '20240520'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')

df_customer_base = pd.DataFrame()
current_dt = start_dt

while current_dt <= end_dt:
    
    current_date_str = current_dt.strftime('%Y%m%d')
    
    customer_base_query = f"""
        SELECT 
            yyyymmdd,
            city_name,
            service_obj_service_name service_name,
            customer_id,
            order_id,
            order_status,
            spd_fraud_flag
        FROM 
            orders.order_logs_snapshot
        WHERE
            yyyymmdd = '{current_date_str}'
            and order_status = 'dropped'
            AND service_obj_service_name IN ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium',
                                            'CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro')
    """
    
    df_day = pd.read_sql(customer_base_query, connection)
    df_customer_base = pd.concat([df_customer_base, df_day], ignore_index=True)
    current_dt += timedelta(days=1)


df_customer_base.to_csv(output_file + '/customer_base_part2.csv', index = False)

In [6]:
# df_customer_base_final.to_csv(output_file + '/customer_base_final.csv', index = False)
df_customer_base_final = pd.read_csv(output_file + '/customer_base_final.csv')

rewards_query = f"""
    with unlocled as (
        SELECT
            yyyymmdd,
            user_id,
            order_id,
            rewards_external_id,
            rewards_type
        FROM 
            (
            SELECT
                yyyymmdd,
                user_id,
                rewards_transaction_id order_id,
                JSON_EXTRACT_SCALAR(rewards, '$.externalRewardMetaData.externalRewardId') rewards_external_id,
                -- rewards_state,
                rewards_type,
                epoch,
                ROW_NUMBER() OVER(PARTITION BY user_id,rewards_transaction_id ORDER BY epoch ) seq_no
                
            FROM 
                raw.kafka_pricing_scratchcards_immutable
            WHERE 
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
                AND rewards_state = 'UNLOCKED'
                AND rewards_type = 'EXTERNAL_REWARD'
                -- AND user_id = '6288c6694281e53bbe33d133'
            GROUP BY 1,2,3,4,5,6
            )
        WHERE
            seq_no = 1
    ),
    
    
    scratched AS (
        
        SELECT
            *
        FROM 
            (
            SELECT
                yyyymmdd,
                user_id,
                rewards_transaction_id order_id,
                rewards_state,
                rewards_type,
                JSON_EXTRACT_SCALAR(rewards, '$.externalRewardMetaData.externalRewardId') rewards_external_id,
                JSON_EXTRACT_SCALAR(rewards, '$.externalReward.partnerName') rewards_partner_name,
                JSON_EXTRACT_SCALAR(rewards, '$.amount') amount,
                epoch,
                ROW_NUMBER() OVER(PARTITION BY user_id,rewards_transaction_id ORDER BY epoch ) seq_no
            FROM 
                raw.kafka_pricing_scratchcards_immutable
            WHERE 
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '20240528'
                -- AND user_id = '6288c6694281e53bbe33d133'
                AND rewards_state = 'SCRATCHED' 
            GROUP BY 1,2,3,4,5,6,7,8,9
            )
        WHERE
            seq_no = 1
        )
        
        SELECT 
            a.*,
            b.rewards_state rewards_state_updated,
            b.rewards_type rewards_type_updated,
            b.rewards_partner_name rewards_partner_name_updated,
            CASE WHEN b.rewards_type = 'COIN' AND b.amount = '0' THEN 'Better luck next time' ELSE b.rewards_partner_name END scratched_state
            
        FROM 
            unlocled a
        LEFT JOIN 
            scratched b
            ON a.yyyymmdd <= b.yyyymmdd
            AND a.user_id = b.user_id
            AND a.order_id = b.order_id

"""

df_rewards_v1 = pd.read_sql(rewards_query, connection)

In [7]:
df_rewards_v1 = pd.read_csv(output_file + '/rewards_v1.csv')

### Mapping

In [8]:
# # Define the mapping 
# mapping = {'Auto': 'Auto',
#            'Auto Pool': 'Auto', 
#            'Link': 'Link', 
#            'Bike Lite': 'Link', 
#            'Cab': 'CabEconomy',} 

# # Apply the mapping to the time_period column 
# df_customer_base['service'] = df_customer_base['service_name'].map(mapping)

## Data Schema

In [9]:
df_customer_base_final.head(2)

,yyyymmdd,city_name,service_name,customer_id,order_id,order_status,spd_fraud_flag
0,20240420,Delhi,CabEconomy,64bbe7e60c4ec46d25f0fb5d,662309cc5273492f21264510,dropped,False
1,20240420,Indore,Auto,662304fdc9dad6b18c25e8a2,6623054564e0ac4b47c7d2bf,dropped,False


In [10]:
df_rewards_v1.head(2)

,yyyymmdd,user_id,order_id,rewards_external_id,rewards_type,rewards_state_updated,rewards_type_updated,rewards_partner_name_updated,scratched_state
0,20240519,6182998316ec5217d5d3de21,6649b2927144a34c0f8e1b8f,6643172e3e1b91fc3591c411,EXTERNAL_REWARD,NaN,NaN,NaN,NaN
1,20240513,64db9895e4eb8c9eb1c544e1,66424f296afdf15098d890a0,6641f70c59683237cd2ba44c,EXTERNAL_REWARD,NaN,NaN,NaN,NaN


In [11]:
df_rewards_v1.scratched_state.unique()

array([nan, 'Better luck next time', 'Rummy Circle', 'My 11Circle',
       'Lenskart', 'OTT Play', 'Myntra', 'Jio Saavn', 'NewSanity1'],
      dtype=object)

In [12]:
df_rewards_v1.nunique()

yyyymmdd                              30
user_id                          6293418
order_id                        16645910
rewards_external_id                   28
rewards_type                           1
rewards_state_updated                  1
rewards_type_updated                   2
rewards_partner_name_updated           7
scratched_state                        8
dtype: int64

### Merge

In [13]:
df_merge = pd.merge(df_customer_base_final,
                    df_rewards_v1,
                    how='left',
                    left_on = ['customer_id', 'order_id'],
                    right_on = ['user_id', 'order_id']
                   )
        
df_merge.head(1)

,yyyymmdd_x,city_name,service_name,customer_id,order_id,order_status,spd_fraud_flag,yyyymmdd_y,user_id,rewards_external_id,rewards_type,rewards_state_updated,rewards_type_updated,rewards_partner_name_updated,scratched_state
0,20240420,Delhi,CabEconomy,64bbe7e60c4ec46d25f0fb5d,662309cc5273492f21264510,dropped,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df_customers = df_merge.groupby(['customer_id']).agg(orders_bins = ('order_id' , 'nunique')).reset_index()
df_customers.shape

(12980809, 2)

In [20]:
df_customers.head(2)

,customer_id,orders_bins
0,5737c6adddbec2203f73316d,1
1,5737c6aeddbec2203f733176,4


In [21]:
df_merged = pd.merge(df_merge,
                     df_customers,
                     how='left',
                     left_on = ['customer_id'],
                     right_on = ['customer_id']
                    )
df_merged.head(1)

,yyyymmdd_x,city_name,service_name,customer_id,order_id,order_status,spd_fraud_flag,yyyymmdd_y,user_id,rewards_external_id,rewards_type,rewards_state_updated,rewards_type_updated,rewards_partner_name_updated,scratched_state,geo-level,orders_bins
0,20240420,Delhi,CabEconomy,64bbe7e60c4ec46d25f0fb5d,662309cc5273492f21264510,dropped,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PAN India,1


In [22]:
df_merge.shape

(48318962, 16)

In [23]:
df_merged.shape

(48318962, 17)

## User Defined func

In [24]:
df_merged['geo-level'] = 'PAN India'

In [29]:
agg_functions = {
    
    'start_date' : ('yyyymmdd_x', 'min'), 
    'end_date' : ('yyyymmdd_x', 'max'),
    'net_customers' : ('customer_id', lambda x: x[df_merged['order_status'] == 'dropped'].nunique()),
    'net_orders' : ('order_id', 
                    lambda x: x[(df_merged.loc[x.index, 'order_status'] == 'dropped')].nunique()),
    
    'sc_unlocked_customers' : ('customer_id', lambda x: x[~df_merged['rewards_external_id'].isna()].nunique()),
    'sc_scratched_customers' : ('customer_id', lambda x: x[df_merged['rewards_state_updated'] == 'SCRATCHED'].nunique()),
    'sc_scratched_earned_customers' : ('customer_id', lambda x: x[df_merged['rewards_type_updated'] == 'EXTERNAL_REWARD'].nunique()),
    
    'sc_unlocked' : ('order_id', lambda x: x[~df_merged['rewards_external_id'].isna()].nunique()),
    'sc_scratched_overall' : ('order_id', lambda x: x[df_merged['rewards_state_updated'] == 'SCRATCHED'].nunique()),
    'sc_scratched_earned' : ('order_id', lambda x: x[df_merged['rewards_type_updated'] == 'EXTERNAL_REWARD'].nunique()),
    'sc_scratched_not_earned' : ('order_id', lambda x: x[df_merged['rewards_type_updated'] == 'COIN'].nunique()),
} 

In [30]:
def calculate_percentage(df, numerator, denominator):
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    return percentage

def calculate_percentage_columns(df, grouper_column):
    
    df['% scratch_rate overall'] = calculate_percentage(df, 'sc_scratched_overall', 'sc_unlocked')
    df['% scratch_rate earned'] = calculate_percentage(df, 'sc_scratched_earned', 'sc_unlocked')
    df['% scratch_rate not_earned'] = calculate_percentage(df, 'sc_scratched_not_earned', 'sc_unlocked')
    df['date_range'] = df['start_date'].astype(str) + ' - ' + df['end_date'].astype(str)
    
    list_agg_column = ['date_range', 
                       'net_customers', 
                       'sc_unlocked_customers', 'sc_scratched_customers', 'sc_scratched_earned_customers',
                       'net_orders', 
                       'sc_unlocked', 'sc_scratched_overall', 'sc_scratched_earned', 'sc_scratched_not_earned',
                       '% scratch_rate overall', '% scratch_rate earned', '% scratch_rate not_earned',]
    grouper_column.remove('geo-level')
    df_new = df[grouper_column + list_agg_column]

    return df_new

## Analysis

In [31]:
df_merged.head(2)

,yyyymmdd_x,city_name,service_name,customer_id,order_id,order_status,spd_fraud_flag,yyyymmdd_y,user_id,rewards_external_id,rewards_type,rewards_state_updated,rewards_type_updated,rewards_partner_name_updated,scratched_state,geo-level,orders_bins
0,20240420,Delhi,CabEconomy,64bbe7e60c4ec46d25f0fb5d,662309cc5273492f21264510,dropped,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PAN India,1
1,20240420,Indore,Auto,662304fdc9dad6b18c25e8a2,6623054564e0ac4b47c7d2bf,dropped,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PAN India,1


### 1. Platform level net rides on given time period?

In [32]:
grouper_column = ['geo-level']

df_overall = df_merged\
.groupby(grouper_column)\
.agg(**agg_functions)\
.sort_values(by=grouper_column)\
.reset_index()

df_overall = calculate_percentage_columns(df_overall, grouper_column)
df_overall

,date_range,net_customers,sc_unlocked_customers,sc_scratched_customers,sc_scratched_earned_customers,net_orders,sc_unlocked,sc_scratched_overall,sc_scratched_earned,sc_scratched_not_earned,% scratch_rate overall,% scratch_rate earned,% scratch_rate not_earned
0,20240420 - 20240520,12980809,6289993,1343558,730700,48318962,16633177,3807513,967358,2840155,22.89,5.82,17.08


Overall Opportunity Sizing
        
    - Customers 
        - Number of platform level net customers - ~1.3 Cr
        - Number of platform level net customers offered rewards (Distributed/Unlocked) - 6.2 M
        - Number of platform level net customers offered rewards (Scratched) - 1.3 M
        
    - Orders
        - Number of platform rides happening - ~4.3 Cr
        - Number of platform rides offered rewards (Distributed/Unlocked) - 1.6 Cr
        - Number of platform rides offered rewards (Scratched) - 3.8 M
        - Number of platform rides offered rewards (Earned) - 0.9 M


    - Benchmark 
        - % scratch_rate earned - 5.82 %
        - % scratch_rate not_earned - 17.08 %
        - % scratch_rate overall - 22.89 %

### 2. Customer level multiple-offer cuts

In [33]:
grouper_column = ['geo-level', 'orders_bins']

df_orders_bins = df_merged\
.groupby(grouper_column)\
.agg(**agg_functions)\
.sort_values(by=grouper_column)\
.reset_index()

df_orders_bins = calculate_percentage_columns(df_orders_bins, grouper_column)
df_orders_bins

,orders_bins,date_range,net_customers,sc_unlocked_customers,sc_scratched_customers,sc_scratched_earned_customers,net_orders,sc_unlocked,sc_scratched_overall,sc_scratched_earned,sc_scratched_not_earned,% scratch_rate overall,% scratch_rate earned,% scratch_rate not_earned
0,1,20240420 - 20240520,5250897,1714295,160731,87437,5250897,1714295,160731,87437,73294,9.38,5.10,4.28
1,2,20240420 - 20240520,2595578,1206328,178895,90164,5191156,1735672,224726,98127,126599,12.95,5.65,7.29
2,3,20240420 - 20240520,1414293,773030,147986,72385,4242879,1417531,223487,83080,140407,15.77,5.86,9.91
3,4,20240420 - 20240520,903220,542919,124088,61040,3612880,1219278,221223,73196,148027,18.14,6.00,12.14
4,5,20240420 - 20240520,603604,386260,100690,50357,3018020,1022613,206942,62857,144085,20.24,6.15,14.09
5,6,20240420 - 20240520,430655,289176,83214,42648,2583930,885766,194231,54904,139327,21.93,6.20,15.73
6,7,20240420 - 20240520,315712,221055,68717,35868,2209984,764464,179269,47596,131673,23.45,6.23,17.22
7,8,20240420 - 20240520,240013,172877,57258,30489,1920104,668336,164346,41440,122906,24.59,6.20,18.39
8,9,20240420 - 20240520,186411,137098,48442,26534,1677699,584829,152245,36936,115309,26.03,6.32,19.72
9,10,20240420 - 20240520,148789,111887,41249,22988,1487890,521294,140520,32768,107752,26.96,6.29,20.67


### 3. Service level net rides on given time period?

In [34]:
grouper_column = ['geo-level', 'service_name']

df_service = df_merged\
.groupby(grouper_column)\
.agg(**agg_functions)\
.sort_values(by=grouper_column)\
.reset_index()

df_service = calculate_percentage_columns(df_service, grouper_column)
df_service

,service_name,date_range,net_customers,sc_unlocked_customers,sc_scratched_customers,sc_scratched_earned_customers,net_orders,sc_unlocked,sc_scratched_overall,sc_scratched_earned,sc_scratched_not_earned,% scratch_rate overall,% scratch_rate earned,% scratch_rate not_earned
0,Auto,20240420 - 20240520,6776488,3630424,818634,413709,19721441,8785475,1958247,524244,1434003,22.29,5.97,16.32
1,Auto NCR,20240420 - 20240520,23489,0,0,0,38892,0,0,0,0,NaN,NaN,NaN
2,Auto Pool,20240422 - 20240520,12383,0,0,0,27033,0,0,0,0,NaN,NaN,NaN
3,Bike Lite,20240420 - 20240520,825614,0,0,0,2499237,0,0,0,0,NaN,NaN,NaN
4,Bike Metro,20240420 - 20240520,90059,0,0,0,152578,0,0,0,0,NaN,NaN,NaN
5,Bike Pink,20240420 - 20240520,2216,0,0,0,2568,0,0,0,0,NaN,NaN,NaN
6,CabEconomy,20240420 - 20240520,2334366,0,0,0,4362240,0,0,0,0,NaN,NaN,NaN
7,Link,20240420 - 20240520,7334687,3491726,722963,353725,21514973,7847702,1849266,443114,1406152,23.56,5.65,17.92


Service level only Link & Auto are distributed by rewards

- Link having 1% higher delta when compared with Auto

## Summary

In [35]:
df_overall

,date_range,net_customers,sc_unlocked_customers,sc_scratched_customers,sc_scratched_earned_customers,net_orders,sc_unlocked,sc_scratched_overall,sc_scratched_earned,sc_scratched_not_earned,% scratch_rate overall,% scratch_rate earned,% scratch_rate not_earned
0,20240420 - 20240520,12980809,6289993,1343558,730700,48318962,16633177,3807513,967358,2840155,22.89,5.82,17.08


In [36]:
df_orders_bins

,orders_bins,date_range,net_customers,sc_unlocked_customers,sc_scratched_customers,sc_scratched_earned_customers,net_orders,sc_unlocked,sc_scratched_overall,sc_scratched_earned,sc_scratched_not_earned,% scratch_rate overall,% scratch_rate earned,% scratch_rate not_earned
0,1,20240420 - 20240520,5250897,1714295,160731,87437,5250897,1714295,160731,87437,73294,9.38,5.10,4.28
1,2,20240420 - 20240520,2595578,1206328,178895,90164,5191156,1735672,224726,98127,126599,12.95,5.65,7.29
2,3,20240420 - 20240520,1414293,773030,147986,72385,4242879,1417531,223487,83080,140407,15.77,5.86,9.91
3,4,20240420 - 20240520,903220,542919,124088,61040,3612880,1219278,221223,73196,148027,18.14,6.00,12.14
4,5,20240420 - 20240520,603604,386260,100690,50357,3018020,1022613,206942,62857,144085,20.24,6.15,14.09
5,6,20240420 - 20240520,430655,289176,83214,42648,2583930,885766,194231,54904,139327,21.93,6.20,15.73
6,7,20240420 - 20240520,315712,221055,68717,35868,2209984,764464,179269,47596,131673,23.45,6.23,17.22
7,8,20240420 - 20240520,240013,172877,57258,30489,1920104,668336,164346,41440,122906,24.59,6.20,18.39
8,9,20240420 - 20240520,186411,137098,48442,26534,1677699,584829,152245,36936,115309,26.03,6.32,19.72
9,10,20240420 - 20240520,148789,111887,41249,22988,1487890,521294,140520,32768,107752,26.96,6.29,20.67


In [38]:
df_orders_bins.head(100).to_clipboard(index=False)